In [ ]:
from atlite.gis import ExclusionContainer, shape_availability, padded_transform_and_shape
import time
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio as rio
from rasterio.enums import Resampling
from numpy import empty
from scipy.ndimage import binary_dilation as dilation
from shapely.geometry import Polygon, box
import shapely
from rasterio.features import geometry_mask

from workflow.scripts.extra_functions import get_bounding

In [ ]:
## method
method = snakemake.params.method # atlite

# input data
# europe_raster = snakemake.input.europe_raster
europe_shp = snakemake.input.europe_shapefile
scenario = snakemake.wildcards.scenario


# params
aggregated_regions = snakemake.params.aggregated_regions
polygon_borders = snakemake.params.polygon_borders
input_scenarios = snakemake.params.input_scenarios["Coastline"]
target_crs = snakemake.params.target_crs # target EPSG

## output data
output_path = snakemake.output.coastline_path

In [ ]:
# resolution
res = 100

# conversion nms to m
nms_to_m = 1852 # 1 nms = 1852 m

# borders
[rectx1, rectx2, recty1, recty2] = polygon_borders
polygon_bounds = get_bounding(target_crs,rectx1,rectx2,recty1,recty2)

# buffer
buffer = input_scenarios[scenario]

In [ ]:
t=time.time()


europe = (
    gpd
    .read_file(europe_shp)
    .replace({"EL": "GR"})
    .query("NUTS_ID == @aggregated_regions")
    .set_index(["NUTS_ID"])
    .loc[:,['geometry']]
    .to_crs("EPSG:4326")
)


polygon = Polygon(
[
    (rectx1, recty1),
    (rectx1, recty2),
    (rectx2, recty2),
    (rectx2, recty1),
    (rectx1, recty1),
]
)

polygon = shapely.segmentize(polygon, max_segment_length=0.5)

europe = (
    gpd
    .clip(europe, polygon)
)
# Remove Jan Mayen
nor = europe.query("NUTS_ID =='NO'").clip(box(0, recty1, rectx2, recty2))
europe = pd.concat([europe.query("NUTS_ID != 'NO'"), nor])
europe = europe.to_crs(target_crs)

onshore = (
    europe
    .assign(area = 1)
    .dissolve(by='area')
    .reset_index()
    .to_crs(target_crs)
)
onshore["area"] = onshore["area"].astype("uint8")

In [ ]:
t=time.time()

if method=="manual":
    dst_transform, dst_shape = padded_transform_and_shape(
            bounds=rio.features.bounds(polygon_bounds),
            res=res # resolution
        )
    raster_onshore = ~geometry_mask(
        onshore.geometry,
        out_shape=dst_shape,
        transform=dst_transform,
        all_touched=True,
        )

    iterations = int(buffer*nms_to_m / res) + 1
    raster_onshore_buffered = dilation(raster_onshore, iterations=iterations)
    
    raster_diff = (raster_onshore_buffered & ~raster_onshore).astype("uint8")

    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_diff.shape[1],
        'height': raster_diff.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': dst_transform,
        'compress': 'lzw'
    }

    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_diff, 1)
    print(time.time()-t)

In [ ]:
t=time.time()

if method=="atlite":
    
    exclusion_container = ExclusionContainer(crs=target_crs)
    exclusion_container.add_geometry(onshore)
    masked, transform = shape_availability(polygon_bounds.geometry, exclusion_container)
    raster_onshore = (~masked)

    exclusion_container = ExclusionContainer(crs=target_crs)
    exclusion_container.add_geometry(onshore, buffer=buffer*nms_to_m)
    masked, transform = shape_availability(polygon_bounds.geometry, exclusion_container)
    raster_onshore_buffered = (~masked)

    raster_diff = (raster_onshore_buffered & ~raster_onshore).astype("uint8")
    
    
    meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'nodata': None,
        'width': raster_diff.shape[1],
        'height': raster_diff.shape[0],
        'count': 1,
        'crs': target_crs,
        'transform': transform,
        'compress': 'lzw'
    }
    
    # Save to file
    with rio.open(output_path, 'w', **meta) as dst:
        dst.write(raster_diff, 1)
    print(time.time()-t)

In [ ]:
if method=="test":
    f = open(output_path, "x")